In [ ]:
import pypsa
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
n = pypsa.Network("../../results/cost_year~2030/transport_cost~custom/demand~1/product~steel/network.nc")

In [ ]:
n.statistics()

In [ ]:
n.statistics.energy_balance()

### Global supply, demand and trade

#### Supply

In [ ]:
supply = n.statistics.energy_balance(comps=["Generator"], groupby=["bus"]).div(1e6)
supply.sort_values(ascending=False)

In [ ]:
print(f"The total supply is {supply.sum():.2f} Mt")

#### Demand

In [ ]:
demand = n.statistics.energy_balance(comps=["Load"], groupby=["bus"]).div(-1e6)
demand.sort_values(ascending=False)

In [ ]:
print(f"The total demand is {demand.sum():.2f} Mt")

#### Trade

In [ ]:
trade = n.statistics.energy_balance(comps=["Link"], groupby=["bus", "carrier"]).div(1e6)
trade

In [ ]:
trade.sum()

Exporters

In [ ]:
exporters = trade[trade < 0] * (-1)
exporters.sort_values(ascending=False)

Importers

In [ ]:
importers = trade[trade > 0]
importers.sort_values(ascending=False)

### Iron ore

In [ ]:
n.statistics.energy_balance(comps=["Link"], groupby=["bus", "carrier"]).div(1e6).loc[:,:,"iron_ore"]

In [ ]:
n.statistics.energy_balance(comps=["Generator"], groupby=["bus", "carrier"]).div(1e6)

### Cost

In [ ]:
print(f"The total cost is {n.objective/1e9:.2f} B EUR")

In [ ]:
n.statistics()

In [ ]:
n.generators.marginal_cost

In [ ]:
n.links.p_nom_opt.sort_values(ascending=False)

### Sankey

### Global map

In [ ]:
plot_config = {"bus_size": 1.0e-7,
               "link_width": 1.5e-8,}

In [ ]:
def plot_trade_network(n):
    # creating color dataframe for type of transportation method
    df_link = n.links.type.astype(str).to_frame()
    df_link.reset_index(inplace=True)
    df_link = df_link.rename(
        columns={df_link.columns[0]: "Link", df_link.columns[1]: "color"}
    )
    # setting all as default to green
    df_link["color"] = "lightgreen"
    # shipping links are changed to blue
    df_link.loc[df_link["Link"].str.contains("shipping"), "color"] = "skyblue"
    df_link.set_index("Link", inplace=True)

    # creating figure
    fig = plt.figure()
    region_gen = n.generators.groupby(["bus"]).p_nom_opt.sum()
    region_load = n.loads.groupby(["bus"]).p_set.sum()
    link_flow = n.links.p_nom_opt.astype(int)
    n.plot(
        bus_sizes=region_load * plot_config["bus_size"],  # 1e-8
        bus_colors="seagreen",
        bus_alpha=1,
        link_widths=0,
        branch_components=["Link"],
    )  # the load at bus in green
    n.plot(
        bus_sizes=region_gen * plot_config["bus_size"],  # 1e-8
        bus_colors="lightsteelblue",
        bus_alpha=0.7,
        link_widths=link_flow * plot_config["link_width"],  # 1e-9
        branch_components=["Link"],
        link_colors=df_link["color"],
    )  # the gen at bus in light blue

    legend_elements = [
        plt.Line2D([0], [0], color="lightgreen", label="pipeline"),
        plt.Line2D([0], [0], color="lightblue", label="shipping"),
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="white",
            label="Demand",
            markerfacecolor="seagreen",
            markersize=10,
        ),
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="white",
            label="Supply",
            markerfacecolor="lightsteelblue",
            markersize=10,
        ),
    ]
    fig.legend(handles=legend_elements, frameon=False)

    # fig.suptitle("scenario:{}-{}-{}".format(snakemake.wildcards["cost_year"],snakemake.wildcards["transport_cost"],snakemake.wildcards["demand"]))
    # fig.savefig(snakemake.output.trade_plot, format="pdf")
    return

In [ ]:
plot_trade_network(n)